### 2026-02-23 ERA5 1deg precipitation

Make a 1deg monthly precipitation file from the 0.25 degree data, following the other coarsened variables.

In [1]:
import xarray as xr
import numpy as np
import xesmf
from dask.diagnostics import ProgressBar
from copy import deepcopy

In [2]:
INPUT_PATH = '../local_data/ERA5/mon/ERA5_total_precipitation_mon_full_sfc_1940-2024.nc'
TEMPLATE_PATH = '../local_data/ERA5/mon_1deg/native6_ERA5_an_v1_Amon_tas_1978-2024.nc'
OUTPUT_PATH = '../local_data/ERA5/mon_1deg/native6_ERA5_an_v1_Amon_pr_1978-2024.nc'
TIME_BOUNDS = ('1979-10-01', '2024-12-31')
LATITUDE_LIMITS = (-87.5, 87.5) # avoid regridding artifacts at the poles
KG_PER_M2_PER_M = 1000
SECONDS_PER_DAY = 86_400

In [3]:
precip_ds = xr.open_dataset(INPUT_PATH, chunks={})
template_ds = xr.open_dataset(TEMPLATE_PATH, chunks={})

/tmp/ipykernel_144028/3370774594.py:2: FutureWarning: In a future version, xarray will not decode the variable 'forecast_period' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  template_ds = xr.open_dataset(TEMPLATE_PATH, chunks={})


In [4]:
destination_grid = (
    template_ds
    .sel(lat=slice(*LATITUDE_LIMITS))
    [['lat', 'lon', 'lat_bnds', 'lon_bnds']]
)
regridder = xesmf.Regridder(
    precip_ds[['lat', 'lon']],
    destination_grid,
    method='conservative'
)
src_dataarray = precip_ds.TP.sel(time=slice(*TIME_BOUNDS))
precip_regridded = regridder.regrid_dataarray(src_dataarray)

/home/brianhenn/miniconda3/envs/fme/lib/python3.11/site-packages/xesmf/backend.py:57: UserWarning: Latitude is outside of [-90, 90]
  warnings.warn('Latitude is outside of [-90, 90]')


In [5]:
# set various attributes to match template
output_ds = deepcopy(template_ds).sel(time=slice(*TIME_BOUNDS))
output_ds['pr'] = xr.full_like(template_ds['tas'], fill_value=np.nan)
output_ds['pr'].loc[{'lat': slice(*LATITUDE_LIMITS)}] = precip_regridded.assign_coords({'time': output_ds.time})
output_ds['pr'] = output_ds['pr'] * KG_PER_M2_PER_M / SECONDS_PER_DAY # units conversion from m (apparently per day) to kg / m ** 2 / s
output_ds['pr'] = output_ds['pr'].assign_attrs({"long_name": "Surface precipitation rate", "units": "kg/m**2/s"})
output_ds = output_ds.drop_vars('tas')

In [6]:
with ProgressBar():
    output_ds.to_netcdf(OUTPUT_PATH)

[########################################] | 100% Completed | 16.87 s


In [7]:
output_ds

<xarray.Dataset> Size: 141MB
Dimensions:               (time: 543, lat: 180, lon: 360, bnds: 2)
Coordinates:
  * time                  (time) datetime64[ns] 4kB 1979-10-17 ... 2024-12-17
  * lat                   (lat) float64 1kB -89.5 -88.5 -87.5 ... 87.5 88.5 89.5
  * lon                   (lon) float64 3kB 0.5 1.5 2.5 ... 357.5 358.5 359.5
    forecast_period       timedelta64[ns] 8B ...
    height                float64 8B ...
    originating_centre    |S50 50B ...
Dimensions without coordinates: bnds
Data variables:
    latitude_longitude    int32 4B ...
    time_bnds             (time, bnds) datetime64[ns] 9kB dask.array<chunksize=(543, 2), meta=np.ndarray>
    lat_bnds              (lat, bnds) float64 3kB dask.array<chunksize=(180, 2), meta=np.ndarray>
    lon_bnds              (lon, bnds) float64 6kB dask.array<chunksize=(360, 2), meta=np.ndarray>
    forecast_period_bnds  (bnds) float64 16B dask.array<chunksize=(2,), meta=np.ndarray>
    pr                    (time, lat, lon) float32 141MB dask.array<chunksize=(543, 180, 360), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.7
    software:     Created with ESMValTool v2.14.0.dev68+g5a317f52f.d20260122